In [221]:
import os
import json
import h5py
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video
import imageio
from statistics import mean
import robomimic.utils.file_utils as FileUtils
from PIL import Image, ImageDraw, ImageFont
import zarr
from diffusion_policy.common.replay_buffer import ReplayBuffer
from filelock import FileLock
from diffusion_policy.codecs.imagecodecs_numcodecs import register_codecs, Jpeg2k
import pdb
from tqdm import tqdm
import xml.etree.ElementTree as ET
import random

In [69]:
og_redcube_data = h5py.File('/proj/vondrick3/sruthi/robots/diffusion_policy/data/robomimic/datasets/lift/ph/robomimic/datasets/lift/ph/image_abs.hdf5', 'r')
# print(og_redcube_data['data'].attrs['env_args'])

# Create Saved Rollout Dataset

In [ ]:
basepath = '/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.01.13/12.45.41_train_diffusion_unet_hybrid_robocasalang_PnPSinkToCounter_trainsplit_imagenet/checkpoints/epoch=0600-val_loss=0.079/PnPSinkToCounter_None_1_13_23_24_6/'
obsdict_robot0_agentview_right_image = np.load(basepath + 'obsdict_robot0_agentview_right_image.npy', 'r')
obsdict_robot0_agentview_left_image = np.load(basepath + 'obsdict_robot0_agentview_left_image.npy', 'r')
obsdict_eyeinhand = np.load(basepath + 'obsdict_eyeinhand.npy', 'r')
obsdict_robot0s = np.load(basepath + 'obsdict_robot0s.npy', 'r')
rewards = np.load(basepath + 'rewards.npy', 'r')
states = np.load(basepath + 'startstates.npy', 'r')
actions = np.load(basepath + 'actions.npy', 'r')
grasping = np.load(basepath + 'grasping.npy', 'r')

In [ ]:
obsdict_robot0_agentview_right_image.shape

In [ ]:
print('hi', basepath)
obsdict_robot0_agentview_right_image = (obsdict_robot0_agentview_right_image.transpose(0,2,1,4,5,3).reshape(13*8,1008,84,84,3)* 255.0).astype(np.uint8)[:-4]
print('obsdict_robot0_agentview_right_image', obsdict_robot0_agentview_right_image.shape)

obsdict_robot0_agentview_left_image = (obsdict_robot0_agentview_left_image.transpose(0,2,1,4,5,3).reshape(13*8,1008,84,84,3)* 255.0).astype(np.uint8)[:-4]
print('obsdict_robot0_agentview_left_image', obsdict_robot0_agentview_left_image.shape)

obsdict_eyeinhand = (obsdict_eyeinhand.transpose(0,2,1,4,5,3).reshape(13*8,1008,84,84,3)* 255.0).astype(np.uint8)[:-4]
print('obsdict_eyeinhand', obsdict_eyeinhand.shape)

obsdict_robot0s = obsdict_robot0s.transpose(0,2,1,3).reshape(13*8,1008,9)[:-4]
print('obsdict_robot0s', obsdict_robot0s.shape)

actions = np.load(trial_hammer_basepath + 'actions.npy', 'r')
print(actions.shape)
actions = actions[:,:,:8,:].transpose(0,2,1,3).reshape(13*8,1008,7)[:-4]
print('actions', actions.shape)

rewards = rewards.transpose(1,0)
print('rewards', rewards.shape)

states = np.repeat(np.array(states)[np.newaxis, :, :], obsdict_agentview.shape[0], axis=0)
print('states', states.shape)

expected:

hi
obsdict_agentview (100, 1008, 84, 84, 3)
obsdict_eyeinhand (100, 1008, 84, 84, 3)
obsdict_robot0s (100, 1008, 9)
(13, 1008, 8, 7)
actions (100, 1008, 7)
rewards (100, 1008)
states (100, 1008, 32)

In [ ]:
test = h5py.File('test.hdf5', 'w')


In [ ]:
test['data'].attrs['temp']=json.dumps({})

In [ ]:
''' CREATE 1 DATASET '''
new_data = {'data':{}}

count = 0
'''
#for high quality dataset
for datapt in range(0,rewards.shape[1]):
    success = int(np.max(rewards[:,datapt]))
    minsuccess = min(np.argwhere(rewards[:,datapt]==1))[0] if success else -1

    if minsuccess>0 and rewards[:,datapt][minsuccess:minsuccess+10].sum()==10:
        stop_demo = minsuccess+16
        new_data['data'][f'demo_{count}'] = {
            'rewards': rewards[:stop_demo,datapt],
            'success': [success],
            'og_pt': [datapt],
            'states': states[:stop_demo,datapt,:],
            'actions': actions[:stop_demo, datapt],
            'obs': {
                'agentview_image': obsdict_agentview[:stop_demo, datapt],
                'robot0_eye_in_hand_image': obsdict_eyeinhand[:stop_demo, datapt],
                'robot0_eef_pos': obsdict_robot0s[:stop_demo, datapt, :3],
                'robot0_eef_quat': obsdict_robot0s[:stop_demo, datapt, 3:7],
                'robot0_gripper_qpos': obsdict_robot0s[:stop_demo, datapt, 7:],
            }
        }
        count+=1 
    else:
        stop_demo = -1
'''

for datapt in range(0,rewards.shape[1]):
    success = np.max(rewards[:,datapt])
    if success>0:
        stop_demo = min(np.argwhere(rewards[:,datapt]==1))[0] + 16
        new_data['data'][f'demo_{count}'] = {
            'rewards': rewards[:stop_demo,datapt],
            'success': [success],
            'og_pt': [datapt],
            'states': states[:stop_demo,datapt,:],
            'actions': actions[:stop_demo, datapt],
            'obs': {
                'agentview_image': obsdict_agentview[:stop_demo, datapt],
                'robot0_eye_in_hand_image': obsdict_eyeinhand[:stop_demo, datapt],
                'robot0_eef_pos': obsdict_robot0s[:stop_demo, datapt, :3],
                'robot0_eef_quat': obsdict_robot0s[:stop_demo, datapt, 3:7],
                'robot0_gripper_qpos': obsdict_robot0s[:stop_demo, datapt, 7:],
            }
        }
        count+=1 
    else:
        stop_demo = -1   
    
# Open HDF5 file and write in the data_dict structure and info
savepath = trial_hammer_basepath+'data_successful_only.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')


datagrp.attrs['env_args'] = og_redcube_data['data'].attrs['env_args']
datagrp.attrs['total'] = len(new_data['data'])


for demo in new_data['data']:
    demogrp = datagrp.create_group(demo)
    
    actionsdset = demogrp.create_dataset('actions', data = new_data['data'][demo]['actions'])
    rewardsdset = demogrp.create_dataset('rewards', data = new_data['data'][demo]['rewards'])
    successdset = demogrp.create_dataset('success', data = new_data['data'][demo]['success'])
    statesdset = demogrp.create_dataset('states', data = new_data['data'][demo]['states'])
    statesdset = demogrp.create_dataset('og_pt', data = new_data['data'][demo]['og_pt'])

    obsgrp = demogrp.create_group('obs') 
    for grp_name in new_data['data'][demo]['obs']:
        dset = obsgrp.create_dataset(grp_name, data = new_data['data'][demo]['obs'][grp_name])
print('demo done', demo)
f.close()


In [ ]:
print(trial_hammer_basepath+'data_all.hdf5')
print(h5py.File(trial_hammer_basepath+'data_alll.hdf5')['data'])
print(h5py.File(trial_hammer_basepath+'data_unsuccessful_only.hdf5')['data'])
print(h5py.File(trial_hammer_basepath+'data_successful_only.hdf5')['data'])

In [ ]:
'''TEST THE NEW DATASET'''

data = h5py.File(trial_hammer_basepath + 'data_all.hdf5', 'r')
print(data['data'])
video_path = trial_hammer_basepath+'temp.mp4'
video_writer = imageio.get_writer(video_path, fps=20)
idx = 50
demo = f'demo_{idx}'
print('shape', data['data'][demo]['obs']['agentview_image'].shape)
print('success', data['data'][demo]['success'][:])
print('ogpt', data['data'][demo]['og_pt'][0])
for b in  data['data'][demo]['obs']['agentview_image']:
    img = Image.fromarray((b).astype(np.uint8))
    d = ImageDraw.Draw(img)
    d.text( (2,2), str(idx), fill=255)
    
    #-- back to array
    b = np.asarray(img)

    video_writer.append_data(b)
    idx+=1
video_writer.close()
Video(video_path, embed=True)
data.close()

# Testing other datasets

In [358]:
data=h5py.File('/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1000-val_loss=0.071/PnPSinkToCounter_mg_val_kbpctk_firsthalf_21984511/datafile.hdf5', 'r')
# print(data['data'].attrs['env_args'])
print(len(data['data']))
# for demo in data['data']:
#     for objs in json.loads(data['data'][demo].attrs['ep_meta'])['object_cfgs']:
#         if objs['name']=='obj':
#             if objs['info']['cat'] not in ['kiwi','bell_pepper','avocado','corn','tangerine']:
#                 print(demo,objs['info']['cat'])


254


In [351]:
list(data['data'].keys())

['demo_0',
 'demo_1',
 'demo_10',
 'demo_100',
 'demo_1000',
 'demo_1001',
 'demo_1002',
 'demo_1003',
 'demo_1004',
 'demo_1005',
 'demo_1006',
 'demo_1007',
 'demo_1008',
 'demo_1009',
 'demo_101',
 'demo_1010',
 'demo_1011',
 'demo_1012',
 'demo_1013',
 'demo_1014',
 'demo_1015',
 'demo_1016',
 'demo_1017',
 'demo_1018',
 'demo_1019',
 'demo_102',
 'demo_1020',
 'demo_1021',
 'demo_1022',
 'demo_1023',
 'demo_1024',
 'demo_1025',
 'demo_1026',
 'demo_1027',
 'demo_1028',
 'demo_1029',
 'demo_103',
 'demo_1030',
 'demo_1031',
 'demo_1032',
 'demo_1033',
 'demo_1034',
 'demo_1035',
 'demo_1036',
 'demo_1037',
 'demo_1038',
 'demo_1039',
 'demo_104',
 'demo_1040',
 'demo_1041',
 'demo_1042',
 'demo_1043',
 'demo_1044',
 'demo_1045',
 'demo_1046',
 'demo_1047',
 'demo_1048',
 'demo_1049',
 'demo_105',
 'demo_1050',
 'demo_1051',
 'demo_1052',
 'demo_1053',
 'demo_1054',
 'demo_1055',
 'demo_1056',
 'demo_1057',
 'demo_1058',
 'demo_1059',
 'demo_106',
 'demo_1060',
 'demo_1061',
 'demo_

In [ ]:
for dirpath, dirnames, filenames in os.walk('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp'):
    for file in filenames:
        if 'mg' in dirpath and file == 'demo_gentex_im128_randcams.hdf5':
            data=h5py.File(os.path.join(dirpath, file),'r')
            objs = set()
            for i in data['data']:
                try:
                    objs.add(json.loads(data['data'][i].attrs['ep_meta'])['lang'].split(' ')[2])
                except:
                    print(i)
            print(dirpath.split('/')[-3], 'n=', len(data['data']), 'uniqueobjs=',len(objs), print(objs))

In [171]:
def get_langset(path):
    data=h5py.File(path,'r')
    demo_keys = sorted(data['data'], key=lambda x: int(x.split('_')[1]))
    langset = set()
    for demo in demo_keys:
        langset.add(json.loads(data['data'][demo].attrs['ep_meta'])['lang'].split('from the sink and place it on the plate located on the counter')[0].split('pick the')[1])
    return langset
print('train langset:', get_langset('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images_train.hdf5'))

print()

print('val langset', get_langset('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images_val.hdf5'))

print('mg_train', get_langset('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images_train.hdf5'))

print('mg_firsthalf_val', get_langset('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images_val_firsthalf.hdf5'))

print('mg_secondhalf_val', get_langset('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images_val_secondhalf.hdf5'))



train langset: {' squash ', ' sweet potato ', ' lemon ', ' potato ', ' fish ', ' banana ', ' apple ', ' carrot ', ' mushroom ', ' lime ', ' peach ', ' tomato ', ' egg ', ' orange ', ' yogurt ', ' mango ', ' broccoli ', ' pear ', ' milk ', ' onion ', ' garlic ', ' steak '}

val langset {' avocado ', ' bell pepper ', ' corn ', ' kiwi ', ' tangerine '}
mg_train {' avocado ', ' squash ', ' eggplant ', ' cucumber ', ' sweet potato ', ' lemon ', ' banana ', ' apple ', ' carrot ', ' mushroom ', ' lime ', ' peach ', ' bell pepper ', ' tomato ', ' egg ', ' cheese ', ' tangerine ', ' orange ', ' mango ', ' yogurt ', ' broccoli ', ' pear ', ' corn ', ' kiwi ', ' onion '}
mg_firsthalf_val {' potato ', ' milk ', ' fish ', ' garlic ', ' steak '}
mg_secondhalf_val {' potato ', ' milk ', ' fish ', ' garlic ', ' steak '}


# Make TRAIN TEST splits based on Objects

In [197]:
'''Take first 300 of the mg train data'''

# Open HDF5 file and write in the data_dict structure and info
base_path = '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images_train_no_kbpckt.hdf5'
current_dataset=h5py.File(base_path, 'r')

savepath = base_path[:-5]+f'_first300.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')
datagrp.attrs['ogdataset'] = base_path
datagrp.attrs['env_args'] = current_dataset['data'].attrs['env_args']

count=0
for demo in current_dataset['data']:
    if count<300:
        demogrp = datagrp.create_group('demo_'+str(count))
        for k in current_dataset['data'][demo].attrs.keys():
            demogrp.attrs[k] = current_dataset['data'][demo].attrs[k]
        demogrp.attrs['og_demo_id']=demo
        actionsdset = demogrp.create_dataset('actions', data = current_dataset['data'][demo]['actions'])
        rewardsdset = demogrp.create_dataset('rewards', data = current_dataset['data'][demo]['rewards'])
        statesdset = demogrp.create_dataset('states', data = current_dataset['data'][demo]['states'])
        donesdset = demogrp.create_dataset('dones', data = current_dataset['data'][demo]['dones'])
        obsgrp = demogrp.create_group('obs') 
        for grp_name in current_dataset['data'][demo]['obs']:
            dset = obsgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['obs'][grp_name])
        actiondictgrp = demogrp.create_group('action_dict') 
        for grp_name in current_dataset['data'][demo]['action_dict']:
            dset = actiondictgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['action_dict'][grp_name])
        print('demo done', demo, count)
        count += 1

print('dataset done', savepath)
f.close()

demo done demo_0 0
demo done demo_1 1
demo done demo_10 2
demo done demo_100 3
demo done demo_1000 4
demo done demo_1001 5
demo done demo_1002 6
demo done demo_1003 7
demo done demo_1004 8
demo done demo_1005 9
demo done demo_1006 10
demo done demo_1007 11
demo done demo_1008 12
demo done demo_1009 13
demo done demo_101 14
demo done demo_1010 15
demo done demo_1011 16
demo done demo_1012 17
demo done demo_1013 18
demo done demo_1014 19
demo done demo_1015 20
demo done demo_1016 21
demo done demo_1017 22
demo done demo_1018 23
demo done demo_1019 24
demo done demo_102 25
demo done demo_1020 26
demo done demo_1021 27
demo done demo_1022 28
demo done demo_1023 29
demo done demo_1024 30
demo done demo_1025 31
demo done demo_1026 32
demo done demo_1027 33
demo done demo_1028 34
demo done demo_1029 35
demo done demo_103 36
demo done demo_1030 37
demo done demo_1031 38
demo done demo_1032 39
demo done demo_1033 40
demo done demo_1034 41
demo done demo_1035 42
demo done demo_1036 43
demo done 

In [ ]:
'''FIRST split the mg dataset into train and val based on objects. these are the objects we want to take OUT OF the train set and put INTO val set: avocado,bell pepper,corn,kiwi,tangerine '''

# Open HDF5 file and write in the data_dict structure and info
base_path = '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images.hdf5'
current_dataset=h5py.File(base_path, 'r')

savepath = base_path[:-5]+f'_train_no_kbpckt.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')
datagrp.attrs['ogdataset'] = base_path
datagrp.attrs['env_args'] = current_dataset['data'].attrs['env_args']
count=0
for demo in current_dataset['data']:
    for objs in json.loads(current_dataset['data'][demo].attrs['ep_meta'])['object_cfgs']:
        if objs['name']=='obj':
            object_id=objs['info']['cat']
    if object_id not in ['avocado','bell_pepper','corn','kiwi','tangerine']:
        demogrp = datagrp.create_group('demo_'+str(count))
        for k in current_dataset['data'][demo].attrs.keys():
            demogrp.attrs[k] = current_dataset['data'][demo].attrs[k]
        demogrp.attrs['og_demo_id']=demo
        actionsdset = demogrp.create_dataset('actions', data = current_dataset['data'][demo]['actions'])
        rewardsdset = demogrp.create_dataset('rewards', data = current_dataset['data'][demo]['rewards'])
        statesdset = demogrp.create_dataset('states', data = current_dataset['data'][demo]['states'])
        donesdset = demogrp.create_dataset('dones', data = current_dataset['data'][demo]['dones'])
        obsgrp = demogrp.create_group('obs') 
        for grp_name in current_dataset['data'][demo]['obs']:
            dset = obsgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['obs'][grp_name])
        actiondictgrp = demogrp.create_group('action_dict') 
        for grp_name in current_dataset['data'][demo]['action_dict']:
            dset = actiondictgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['action_dict'][grp_name])
        print('demo done', demo, count)
        count += 1

print('dataset done', savepath)
f.close()

In [188]:
'''FIRST split the mg dataset into train and val based on objects. these are the objects we want to take OUT OF the train set and put INTO val set: avocado,bell pepper,corn,kiwi,tangerine '''

# Open HDF5 file and write in the data_dict structure and info
base_path = '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images.hdf5'
current_dataset=h5py.File(base_path, 'r')

savepath = base_path[:-5]+f'_val_kbpckt.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')
datagrp.attrs['ogdataset'] = base_path
datagrp.attrs['env_args'] = current_dataset['data'].attrs['env_args']
count=0
for demo in current_dataset['data']:
    for objs in json.loads(current_dataset['data'][demo].attrs['ep_meta'])['object_cfgs']:
        if objs['name']=='obj':
            object_id=objs['info']['cat']
    print(object_id)
    if object_id in ['avocado','bell_pepper','corn','kiwi','tangerine']:
        demogrp = datagrp.create_group('demo_'+str(count))
        for k in current_dataset['data'][demo].attrs.keys():
            demogrp.attrs[k] = current_dataset['data'][demo].attrs[k]
        demogrp.attrs['og_demo_id']=demo
        actionsdset = demogrp.create_dataset('actions', data = current_dataset['data'][demo]['actions'])
        rewardsdset = demogrp.create_dataset('rewards', data = current_dataset['data'][demo]['rewards'])
        statesdset = demogrp.create_dataset('states', data = current_dataset['data'][demo]['states'])
        donesdset = demogrp.create_dataset('dones', data = current_dataset['data'][demo]['dones'])
        obsgrp = demogrp.create_group('obs') 
        for grp_name in current_dataset['data'][demo]['obs']:
            dset = obsgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['obs'][grp_name])
        actiondictgrp = demogrp.create_group('action_dict') 
        for grp_name in current_dataset['data'][demo]['action_dict']:
            dset = actiondictgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['action_dict'][grp_name])
        print('demo done', demo, count)
        count += 1

print('dataset done', savepath)
f.close()


avocado
demo done demo_0 0
orange
orange
tomato
banana
eggplant
cucumber
mango
carrot
orange
onion
sweet_potato
mango
kiwi
demo done demo_1009 1
cucumber
tomato
mango
lime
sweet_potato
corn
demo done demo_1014 2
sweet_potato
apple
squash
kiwi
demo done demo_1018 3
onion
tomato
corn
demo done demo_1020 4
peach
carrot
orange
tomato
cucumber
onion
tomato
garlic
eggplant
squash
potato
potato
peach
bell_pepper
demo done demo_1033 5
carrot
lime
lemon
tangerine
demo done demo_1037 6
steak
egg
potato
garlic
cucumber
tomato
steak
corn
demo done demo_1044 7
lime
mango
apple
bell_pepper
demo done demo_1048 8
lime
onion
tomato
mushroom
peach
cucumber
broccoli
fish
apple
lemon
tomato
apple
broccoli
carrot
cheese
mango
lime
tangerine
demo done demo_1064 9
cheese
yogurt
onion
pear
kiwi
demo done demo_1069 10
broccoli
corn
demo done demo_1070 11
avocado
demo done demo_1071 12
eggplant
carrot
apple
onion
pear
broccoli
apple
lime
onion
tangerine
demo done demo_1080 13
broccoli
garlic
broccoli
cucumber
c

In [195]:
h5py.File('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images_val_kbpckt_firsthalf.hdf5')['data']

<HDF5 group "/data" (253 members)>

In [189]:
'''MAKE FIRST HALF and SECOND HALF of val set'''

# Open HDF5 file and write in the data_dict structure and info
base_path = '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images_val_kbpckt.hdf5'
all_objects={}
current_dataset=h5py.File(base_path, 'r')
for x in current_dataset['data']:
    try:
        for objs in json.loads(current_dataset['data'][x].attrs['ep_meta'])['object_cfgs']:
            if objs['name']=='obj':
                if objs['info']['cat'] not in all_objects:
                    all_objects[objs['info']['cat']]=[x]
                else:
                    all_objects[objs['info']['cat']].append(x)
    except:
        print('issue:', x)

all_objects = dict(sorted(all_objects.items(), key=lambda item: item[1]))
# print(json.dumps(all_objects, indent=4))

#SAVE THE FIRST HALF
savepath = base_path[:-5]+f'_firsthalf.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')
datagrp.attrs['ogdataset'] = base_path
datagrp.attrs['env_args'] = current_dataset['data'].attrs['env_args']

count=0
for object_class, object_class_demos in all_objects.items():
    length_object_class_demos=len(object_class_demos)
    for demo in object_class_demos[:int(length_object_class_demos/2)]:
        demogrp = datagrp.create_group('demo_'+str(count))
        for k in current_dataset['data'][demo].attrs.keys():
            demogrp.attrs[k] = current_dataset['data'][demo].attrs[k]
        demogrp.attrs['og_demo_id']=demo
        actionsdset = demogrp.create_dataset('actions', data = current_dataset['data'][demo]['actions'])
        rewardsdset = demogrp.create_dataset('rewards', data = current_dataset['data'][demo]['rewards'])
        statesdset = demogrp.create_dataset('states', data = current_dataset['data'][demo]['states'])
        donesdset = demogrp.create_dataset('dones', data = current_dataset['data'][demo]['dones'])
        obsgrp = demogrp.create_group('obs') 
        for grp_name in current_dataset['data'][demo]['obs']:
            dset = obsgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['obs'][grp_name])
        actiondictgrp = demogrp.create_group('action_dict') 
        for grp_name in current_dataset['data'][demo]['action_dict']:
            dset = actiondictgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['action_dict'][grp_name])
        print('demo done', demo, count)
        count += 1

print('dataset done', savepath)
f.close()



#SAVE THE SECOND HALF
savepath = base_path[:-5]+f'_secondhalf.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')
datagrp.attrs['ogdataset'] = base_path
datagrp.attrs['env_args'] = current_dataset['data'].attrs['env_args']

count=0
for object_class, object_class_demos in all_objects.items():
    length_object_class_demos=len(object_class_demos)
    for demo in object_class_demos[int(length_object_class_demos/2):]:
        demogrp = datagrp.create_group('demo_'+str(count))
        for k in current_dataset['data'][demo].attrs.keys():
            demogrp.attrs[k] = current_dataset['data'][demo].attrs[k]
        demogrp.attrs['og_demo_id']=demo
        actionsdset = demogrp.create_dataset('actions', data = current_dataset['data'][demo]['actions'])
        rewardsdset = demogrp.create_dataset('rewards', data = current_dataset['data'][demo]['rewards'])
        statesdset = demogrp.create_dataset('states', data = current_dataset['data'][demo]['states'])
        donesdset = demogrp.create_dataset('dones', data = current_dataset['data'][demo]['dones'])
        obsgrp = demogrp.create_group('obs') 
        for grp_name in current_dataset['data'][demo]['obs']:
            dset = obsgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['obs'][grp_name])
        actiondictgrp = demogrp.create_group('action_dict') 
        for grp_name in current_dataset['data'][demo]['action_dict']:
            dset = actiondictgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['action_dict'][grp_name])
        print('demo done', demo, count)
        count += 1

print('dataset done', savepath)
f.close()


demo done demo_0 0
demo done demo_106 1
demo done demo_117 2
demo done demo_12 3
demo done demo_121 4
demo done demo_130 5
demo done demo_132 6
demo done demo_138 7
demo done demo_139 8
demo done demo_145 9
demo done demo_147 10
demo done demo_149 11
demo done demo_152 12
demo done demo_155 13
demo done demo_157 14
demo done demo_160 15
demo done demo_164 16
demo done demo_165 17
demo done demo_172 18
demo done demo_173 19
demo done demo_174 20
demo done demo_180 21
demo done demo_183 22
demo done demo_185 23
demo done demo_186 24
demo done demo_193 25
demo done demo_203 26
demo done demo_212 27
demo done demo_214 28
demo done demo_238 29
demo done demo_24 30
demo done demo_240 31
demo done demo_243 32
demo done demo_245 33
demo done demo_248 34
demo done demo_25 35
demo done demo_253 36
demo done demo_254 37
demo done demo_262 38
demo done demo_267 39
demo done demo_279 40
demo done demo_28 41
demo done demo_282 42
demo done demo_286 43
demo done demo_289 44
demo done demo_296 45
demo

# Combine Datasets

# These are the tasks we are combingin into 1 dataset:


In [227]:
paths = [
"/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images_train_no_kbpckt.hdf5",
"/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1000-val_loss=0.071/PnPSinkToCounter_mg_train_no_kbpctk_21819416/datafile.hdf5", 
"/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1000-val_loss=0.071/PnPSinkToCounter_mg_val_kbpctk_firsthalf_21984511/datafile.hdf5"
]
print(*paths, sep="\n")

/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images_train_no_kbpckt.hdf5
/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1000-val_loss=0.071/PnPSinkToCounter_mg_train_no_kbpctk_21819416/datafile.hdf5
/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1000-val_loss=0.071/PnPSinkToCounter_mg_val_kbpctk_firsthalf_21984511/datafile.hdf5


In [ ]:
"[\"/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images_train_no_kbpckt.hdf5\",
\"/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1000-val_loss=0.071/PnPSinkToCounter_mg_train_no_kbpctk_21819416/datafile.hdf5\", 
\"/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1000-val_loss=0.071/PnPSinkToCounter_mg_val_kbpctk_firsthalf_21984511/datafile.hdf5\"]"

In [234]:
h5py.File('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images_val_kbpckt_firsthalf.hdf5')['data']

<HDF5 group "/data" (253 members)>

In [229]:
''' CREATE 1 DATASET '''
# Open HDF5 file and write in the data_dict structure and info
savepath = '/proj/vondrick3/sruthi/robots/diffusion_policy/data/big_classifier_data_combined.hdf5'
testdata = h5py.File(paths[0], 'r')
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')
datagrp.attrs['datasets_included'] = ', '.join(paths)
datagrp.attrs['env_args'] = testdata['data'].attrs['env_args']
count=0
for current_path in paths:
    current_dataset = h5py.File(current_path)
    for demo in current_dataset['data']:
        demogrp = datagrp.create_group('demo_'+str(count))
        for k in current_dataset['data'][demo].attrs.keys():
            demogrp.attrs[k] = current_dataset['data'][demo].attrs[k]
        demogrp.attrs['og_demo_id']=demo
        demogrp.attrs['dataset_path']=current_path
        actionsdset = demogrp.create_dataset('actions', data = current_dataset['data'][demo]['actions'])
        rewardsdset = demogrp.create_dataset('rewards', data = current_dataset['data'][demo]['rewards'])
        statesdset = demogrp.create_dataset('states', data = current_dataset['data'][demo]['states'])
        # donesdset = demogrp.create_dataset('dones', data = current_dataset['data'][demo]['dones'])
        obsgrp = demogrp.create_group('obs') 
        for grp_name in current_dataset['data'][demo]['obs']:
            dset = obsgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['obs'][grp_name])
        # actiondictgrp = demogrp.create_group('action_dict') 
        # for grp_name in current_dataset['data'][demo]['action_dict']:
        #     dset = actiondictgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['action_dict'][grp_name])
        print('demo done', count)
        count += 1

    print('dataset done', current_path)
f.close()

demo done 0
demo done 1
demo done 2
demo done 3
demo done 4
demo done 5
demo done 6
demo done 7
demo done 8
demo done 9
demo done 10
demo done 11
demo done 12
demo done 13
demo done 14
demo done 15
demo done 16
demo done 17
demo done 18
demo done 19
demo done 20
demo done 21
demo done 22
demo done 23
demo done 24
demo done 25
demo done 26
demo done 27
demo done 28
demo done 29
demo done 30
demo done 31
demo done 32
demo done 33
demo done 34
demo done 35
demo done 36
demo done 37
demo done 38
demo done 39
demo done 40
demo done 41
demo done 42
demo done 43
demo done 44
demo done 45
demo done 46
demo done 47
demo done 48
demo done 49
demo done 50
demo done 51
demo done 52
demo done 53
demo done 54
demo done 55
demo done 56
demo done 57
demo done 58
demo done 59
demo done 60
demo done 61
demo done 62
demo done 63
demo done 64
demo done 65
demo done 66
demo done 67
demo done 68
demo done 69
demo done 70
demo done 71
demo done 72
demo done 73
demo done 74
demo done 75
demo done 76
demo done

In [232]:
''' shuffle 1 DATASET '''
# Open HDF5 file and write in the data_dict structure and info
current_dataset_path='/proj/vondrick3/sruthi/robots/diffusion_policy/data/big_classifier_data_combined.hdf5'
current_dataset = h5py.File(current_dataset_path,'r')
shuffled_dataset_path = '/proj/vondrick3/sruthi/robots/diffusion_policy/data/big_classifier_data_combined_shuffled.hdf5'
f = h5py.File(shuffled_dataset_path, 'w')
datagrp = f.create_group('data')
datagrp.attrs['datasets_included'] = current_dataset_path
datagrp.attrs['env_args'] = current_dataset['data'].attrs['env_args']
count=0
list_of_demos= list(current_dataset['data'].keys())
random.shuffle(list_of_demos)
print(list_of_demos[:100])
for demo in list_of_demos:
    demogrp = datagrp.create_group('demo_'+str(count))
    for k in current_dataset['data'][demo].attrs.keys():
        demogrp.attrs[k] = current_dataset['data'][demo].attrs[k]
    demogrp.attrs['og_demo_id']=demo
    demogrp.attrs['dataset_path']=current_path
    actionsdset = demogrp.create_dataset('actions', data = current_dataset['data'][demo]['actions'])
    rewardsdset = demogrp.create_dataset('rewards', data = current_dataset['data'][demo]['rewards'])
    statesdset = demogrp.create_dataset('states', data = current_dataset['data'][demo]['states'])
    # donesdset = demogrp.create_dataset('dones', data = current_dataset['data'][demo]['dones'])
    obsgrp = demogrp.create_group('obs') 
    for grp_name in current_dataset['data'][demo]['obs']:
        dset = obsgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['obs'][grp_name])
    # actiondictgrp = demogrp.create_group('action_dict') 
    # for grp_name in current_dataset['data'][demo]['action_dict']:
    #     dset = actiondictgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['action_dict'][grp_name])
    print('demo done', count)
    count += 1
f.close()
print('shuffling done', shuffled_dataset_path)

['demo_1426', 'demo_3961', 'demo_3413', 'demo_1258', 'demo_1496', 'demo_4353', 'demo_4224', 'demo_2031', 'demo_1633', 'demo_3474', 'demo_3903', 'demo_5124', 'demo_3897', 'demo_4912', 'demo_3263', 'demo_2657', 'demo_792', 'demo_1451', 'demo_2637', 'demo_1046', 'demo_1404', 'demo_627', 'demo_4178', 'demo_4542', 'demo_275', 'demo_72', 'demo_2786', 'demo_3802', 'demo_3335', 'demo_2963', 'demo_2592', 'demo_2952', 'demo_2821', 'demo_1779', 'demo_4439', 'demo_2735', 'demo_106', 'demo_2958', 'demo_1444', 'demo_4867', 'demo_3162', 'demo_2912', 'demo_4378', 'demo_1628', 'demo_1223', 'demo_4546', 'demo_296', 'demo_213', 'demo_3204', 'demo_1354', 'demo_1631', 'demo_1918', 'demo_578', 'demo_925', 'demo_3359', 'demo_4073', 'demo_5220', 'demo_4665', 'demo_1974', 'demo_812', 'demo_4883', 'demo_2247', 'demo_4309', 'demo_1064', 'demo_4505', 'demo_3385', 'demo_957', 'demo_4649', 'demo_4896', 'demo_3259', 'demo_4899', 'demo_2871', 'demo_613', 'demo_760', 'demo_1784', 'demo_1038', 'demo_3390', 'demo_1757',

In [ ]:
'''TEST THE NEW DATASET'''

data = h5py.File(savepath, 'r')
print(data['data'])
video_path = base_path+'temp1.mp4'
video_writer = imageio.get_writer(video_path, fps=20)
idx = 6040
demo = f'demo_{idx}'
print(data['data'][demo]['object'])
print(data['data'][demo]['obs']['agentview_image'].shape)
print(data['data'][demo]['success'][:])
for b in  data['data'][demo]['obs']['agentview_image']:
    img = Image.fromarray((b).astype(np.uint8))
    d = ImageDraw.Draw(img)
    d.text( (2,2), str(idx), fill=255)
    
    
    #-- back to array
    b = np.asarray(img)

    video_writer.append_data(b)
    idx+=1
video_writer.close()
Video(video_path, embed=True)
data.close()